In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/mpwolke/just-you-wait-rishi-sunak/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_5_try_0.pickle

In [ ]:
%%PandasProfile
### cell 6 ###

# Instantiate stemmer and stop-words set once
stemmer = nltk.stem.snowball.SnowballStemmer('english')
stop_words = set(stopwords.words('english'))

# 1) Vectorized, C-level string ops for line breaks, punctuation/special chars and lowercasing
s = (
    train['text']
        .str.replace(r'[\r\n]+', ' ', regex=True)
        .str.replace(r'[^A-Za-z0-9\s]', ' ', regex=True)
        .str.lower()
)

# 2) Token-level processing: split, drop 1-char tokens & stopwords, stem
def _token_process(text):
    tokens = text.split()
    return ' '.join(
        stemmer.stem(tok)
        for tok in tokens
        if len(tok) > 1 and tok not in stop_words
    )

# 3) Apply per-row token processing and vectorized digit removal
train['clean_text'] = (
    s.map(_token_process)
     .str.replace(r'\d+', '', regex=True)
)